
# # 03_s4 — Compute log-signatures of Neural SDE generated paths
#
 **Job.** Convert `neural_sde_paths.parquet` (wide raw-path schema) into
 `logsigs_s4.parquet` with the same representation as `logsigs_s3.parquet`
 so that Notebook 05 can compare Stream 4 against Stream 3 like-for-like.

 **Same conventions as 03_s3** (verbatim): segment-standardise → lead-lag
 transform on price → time channel → 4D path of length 2*(T-1)=142
 iisignature.logsig at depth 2 → 10-dim logsig vector.

 **Conditioning columns** (`cond_*`, `regime_*`) are pulled directly from
 `logsigs_s3.parquet` by `segment_id` join. Identical conditioning between
 Stream 3 and Stream 4 logsig files is required for any conditioning-aware
 metric in 05 to be comparable across streams.

In [1]:
#  [Cell 2 — Setup]
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path
!pip install iisignature
import numpy as np
import pandas as pd
import iisignature

ROOT      = Path('/content/drive/MyDrive/energy_synthetic_data')   # adjust if needed
DATA_DIR  = ROOT / 'data'
OUT_DIR   = ROOT / 'outputs' / 'stream4_neural_sde'

PATH_NSDE_PATHS = ROOT / 'outputs' / 'stream4_neural_sde' / 'neural_sde_paths.parquet'
PATH_LOGSIG_S3  = DATA_DIR / 'logsigs_s3.parquet'
PATH_OUT        = DATA_DIR / 'logsigs_s4.parquet'
PATH_OUT_STATS  = DATA_DIR / 'logsigs_s4_stats.json'

SEGMENT_LENGTH = 72
SIG_DEPTH      = 4
USE_LEAD_LAG   = True

# After lead-lag of a length-T price channel + time + wind interp:
PATH_LEN = 2 * (SEGMENT_LENGTH - 1)   # 142
PATH_DIM = 4                           # [t, ll_lead, ll_lag, w_ll]
LOGSIG_DIM = iisignature.logsiglength(PATH_DIM, SIG_DEPTH)  # 10

print(f"Path dim: {PATH_DIM}, Path length (post lead-lag): {PATH_LEN}")
print(f"Sig depth: {SIG_DEPTH}, Logsig dim: {LOGSIG_DIM}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for iisignature: filename=iisignature-0.24-cp312-cp312-linux_x86_64.whl size=3246770 sha256=fb6568ec7daac82a88d893e5eee6ae80a94314b7df3cdc417a613aff78da94a3
  Stored in directory: /root/.cache/pip/wheels/7d/b3/8c/168fb7d3b31255a501dea7fa1f5e5098e63ec61a786eeb0156
Successfully built iisignature
Path dim: 4, Path length (post lead-lag): 142
Sig depth: 4, Logsig dim: 90


In [2]:
# [Cell 3 — Helpers (verbatim copy from 03_s3 for representation parity)]

def lead_lag_transform(x: np.ndarray) -> np.ndarray:
    T = len(x)
    path = np.zeros((2 * (T - 1), 2), dtype=np.float64)
    for k in range(T - 1):
        path[2 * k]     = [x[k],     x[k]]
        path[2 * k + 1] = [x[k + 1], x[k]]
    return path


def safe_standardise_within_segment(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isfinite(x).any() else 0.0)
    mu = np.nanmean(x); sd = np.nanstd(x)
    if not np.isfinite(sd) or sd < 1e-8:
        return x - mu
    return (x - mu) / sd


def build_path(price_seg: np.ndarray, wind_seg: np.ndarray, use_lead_lag: bool = True) -> np.ndarray:
    p_norm = safe_standardise_within_segment(price_seg)
    w_norm = safe_standardise_within_segment(wind_seg)
    if use_lead_lag:
        ll = lead_lag_transform(p_norm)
        t_ll = np.linspace(0.0, 1.0, len(ll))
        w_ll = np.interp(t_ll, np.linspace(0.0, 1.0, len(w_norm)), w_norm)
        return np.column_stack([t_ll, ll, w_ll])
    t = np.linspace(0.0, 1.0, len(p_norm))
    return np.column_stack([t, p_norm, w_norm])

In [3]:
# [Cell 4 — Load Neural SDE generated paths]
df_nsde = pd.read_parquet(PATH_NSDE_PATHS)
print(f"neural_sde_paths.parquet: {df_nsde.shape}")

# Required columns from Cell 15 of 04_s4
PRICE_COLS = [f'price_t{t:03d}' for t in range(SEGMENT_LENGTH)]
WIND_COLS  = [f'wind_t{t:03d}'  for t in range(SEGMENT_LENGTH)]
META_COLS  = ['segment_id', 'start_time', 'start_idx', 'end_idx', 'split', 'regime']
STAT_COLS  = ['price_mean', 'price_std', 'wind_mean', 'wind_std']

missing = [c for c in PRICE_COLS + WIND_COLS + META_COLS if c not in df_nsde.columns]
assert not missing, f"Missing columns in neural_sde_paths: {missing[:8]}{'...' if len(missing) > 8 else ''}"

print(f"Splits: {df_nsde['split'].value_counts().to_dict()}")
print(f"Regimes: {df_nsde['regime'].value_counts().head(5).to_dict()}")

neural_sde_paths.parquet: (6501, 154)
Splits: {'train': 4566, 'val': 971, 'test': 964}
Regimes: {'volatile': 1824, 'negative': 910, 'spike_moderate_sustained': 761, 'calm_mid_wind_calm': 726, 'spike_extreme_sustained': 708}


In [4]:
!ls /content/drive/MyDrive/energy_synthetic_data/outputs/stream4_neural_sde/

04_s4_training_curves.png  neural_sde_model.pt
MODEL_CARD.md		   neural_sde_paths.parquet


In [5]:
#  [Cell 5 — Reconstruct (price, wind) cube from wide schema]
N = len(df_nsde)
price_cube = df_nsde[PRICE_COLS].to_numpy(dtype=np.float64)   # (N, 72)
wind_cube  = df_nsde[WIND_COLS].to_numpy(dtype=np.float64)    # (N, 72)

assert price_cube.shape == (N, SEGMENT_LENGTH)
assert wind_cube.shape  == (N, SEGMENT_LENGTH)
assert np.isfinite(price_cube).all(), "Non-finite values in price paths"
assert np.isfinite(wind_cube).all(),  "Non-finite values in wind paths"

print(f"Reconstructed cubes: price {price_cube.shape}, wind {wind_cube.shape}")
print(f"Price (input, already standardised) — mean={price_cube.mean():+.4f}, std={price_cube.std():.4f}")
print(f"Wind  (input, already standardised) — mean={wind_cube.mean():+.4f}, std={wind_cube.std():.4f}")

Reconstructed cubes: price (6501, 72), wind (6501, 72)
Price (input, already standardised) — mean=-0.2698, std=2.7479
Wind  (input, already standardised) — mean=+0.4461, std=3.3149


In [6]:
#  [Cell 6 — Compute logsigs via build_path + iisignature]
sig_spec = iisignature.prepare(PATH_DIM, SIG_DEPTH)

# Sanity test on first segment to confirm shapes line up
test_path = build_path(price_cube[0], wind_cube[0], use_lead_lag=USE_LEAD_LAG)
assert test_path.shape == (PATH_LEN, PATH_DIM), \
    f"Path shape mismatch: got {test_path.shape}, expected ({PATH_LEN}, {PATH_DIM})"
test_logsig = iisignature.logsig(test_path, sig_spec)
assert test_logsig.shape == (LOGSIG_DIM,), f"Logsig shape: {test_logsig.shape}"
print(f"Sanity: path shape {test_path.shape}, logsig shape {test_logsig.shape}")

# Compute for all segments
logsigs = np.zeros((N, LOGSIG_DIM), dtype=np.float64)
for i in range(N):
    p = build_path(price_cube[i], wind_cube[i], use_lead_lag=USE_LEAD_LAG)
    logsigs[i] = iisignature.logsig(p, sig_spec)
    if (i + 1) % 1000 == 0 or i == N - 1:
        print(f"  computed {i+1}/{N}")

assert np.isfinite(logsigs).all(), "Non-finite values in computed logsigs"

ls_cols = [f'ls_{j}' for j in range(LOGSIG_DIM)]
df_ls = pd.DataFrame(logsigs, columns=ls_cols)
print(f"\nLogsig matrix: {logsigs.shape}")
print(f"Per-feature std: {logsigs.std(axis=0).round(3)}")

Sanity: path shape (142, 4), logsig shape (90,)
  computed 1000/6501
  computed 2000/6501
  computed 3000/6501
  computed 4000/6501
  computed 5000/6501
  computed 6000/6501
  computed 6501/6501

Logsig matrix: (6501, 90)
Per-feature std: [0.    2.742 2.707 2.544 0.7   0.681 0.739 2.585 3.47  4.134 0.085 0.081
 0.095 0.544 0.554 0.652 1.045 0.515 0.663 0.729 0.706 0.521 3.431 2.885
 3.283 4.637 3.611 2.517 3.499 3.007 0.013 0.013 0.014 0.093 0.092 0.136
 0.18  0.089 0.139 0.145 0.141 0.103 0.025 0.108 0.475 1.022 0.484 1.438
 0.638 0.842 0.858 0.764 0.548 0.109 1.366 1.396 0.843 1.338 0.446 0.629
 0.883 0.877 0.608 0.635 0.717 0.706 1.226 0.604 0.743 0.678 0.656 0.478
 3.109 2.167 5.854 5.063 4.587 2.312 3.9   3.004 3.981 6.573 3.896 3.341
 4.186 2.76  1.775 2.804 2.862 2.284]


In [7]:
#  [Cell 7 — Pull conditioning columns from logsigs_s3.parquet]
df_s3 = pd.read_parquet(PATH_LOGSIG_S3)
cond_cols   = [c for c in df_s3.columns if c.startswith('cond_')]
regime_cols = [c for c in df_s3.columns if c.startswith('regime_') and c != 'regime']
print(f"From logsigs_s3.parquet: {len(cond_cols)} cond_* cols, {len(regime_cols)} regime_* cols")
assert len(cond_cols) + len(regime_cols) == 19, \
    f"Expected 19-d conditioning, got {len(cond_cols) + len(regime_cols)}"

cols_to_pull = ['segment_id'] + cond_cols + regime_cols
if 'conditioning_dim' in df_s3.columns:
    cols_to_pull.append('conditioning_dim')
df_cond = df_s3[cols_to_pull]

# Inner merge to be safe; verify no segments lost
n_before = len(df_nsde)
df_meta = df_nsde[META_COLS + STAT_COLS].copy()
df_meta_with_cond = df_meta.merge(df_cond, on='segment_id', how='inner')
n_after = len(df_meta_with_cond)
print(f"\nMerge: {n_before} → {n_after}  (dropped {n_before - n_after})")
assert n_after == n_before, "Lost segments in conditioning join — segment_id mismatch?"

# Re-align logsig matrix to merge order (merge can reorder)
order_map = {sid: i for i, sid in enumerate(df_nsde['segment_id'].to_list())}
order_idx = df_meta_with_cond['segment_id'].map(order_map).to_numpy()
df_ls_ordered = df_ls.iloc[order_idx].reset_index(drop=True)

df_out = pd.concat([df_meta_with_cond.reset_index(drop=True), df_ls_ordered], axis=1)
print(f"\nFinal df_out shape: {df_out.shape}")
print(f"Columns: {len(META_COLS)} meta + {len(STAT_COLS)} stats + "
      f"{len(cond_cols) + len(regime_cols)} cond + {LOGSIG_DIM} ls = "
      f"{len(META_COLS) + len(STAT_COLS) + len(cond_cols) + len(regime_cols) + LOGSIG_DIM}"
      f"{' (+1 for conditioning_dim)' if 'conditioning_dim' in df_out.columns else ''}")

From logsigs_s3.parquet: 10 cond_* cols, 9 regime_* cols

Merge: 6501 → 6501  (dropped 0)

Final df_out shape: (6501, 120)
Columns: 6 meta + 4 stats + 19 cond + 90 ls = 119 (+1 for conditioning_dim)


In [8]:
# [Cell 8 — Sanity checks vs logsigs_s3]
# Compare distributional properties of Stream 4 logsigs vs Stream 3 (real) logsigs.
# Identical paths would give identical logsigs; we expect SIMILAR but not identical.
# Anything wildly off-scale is a bug in the path construction.

ls_cols_s3 = [c for c in df_s3.columns if c.startswith('ls_')]
assert len(ls_cols_s3) == LOGSIG_DIM, f"S3 logsig dim {len(ls_cols_s3)} != {LOGSIG_DIM}"

# Restrict to common segment_ids for like-for-like comparison
common_ids = set(df_out['segment_id']) & set(df_s3['segment_id'])
m_s4 = df_out[df_out['segment_id'].isin(common_ids)].copy()
m_s3 = df_s3[df_s3['segment_id'].isin(common_ids)].copy()
m_s3 = m_s3.set_index('segment_id').loc[m_s4['segment_id']].reset_index()

print(f"Common segments for comparison: {len(common_ids)}")
print(f"\n{'feat':<6} {'real_mean':>10} {'s4_mean':>10} {'real_std':>10} {'s4_std':>10} {'std_ratio':>10}")
for j, lc in enumerate(ls_cols):
    r_mu = m_s3[ls_cols_s3[j]].mean(); r_sd = m_s3[ls_cols_s3[j]].std()
    s_mu = m_s4[lc].mean();             s_sd = m_s4[lc].std()
    ratio = s_sd / r_sd if r_sd > 1e-12 else float('nan')
    print(f"{lc:<6} {r_mu:>+10.3f} {s_mu:>+10.3f} {r_sd:>10.3f} {s_sd:>10.3f} {ratio:>10.3f}")

# Per-split count check — must match what came in from neural_sde_paths
print(f"\nSplit counts in Stream 4 logsigs: {df_out['split'].value_counts().to_dict()}")

Common segments for comparison: 6501

feat    real_mean    s4_mean   real_std     s4_std  std_ratio
ls_0       +1.000     +1.000      0.000      0.000        nan
ls_1       -0.033     +0.356      1.442      2.742      1.902
ls_2       +0.086     +0.330      1.426      2.707      1.898
ls_3       -0.250     +0.722      2.365      2.545      1.076
ls_4       -0.203     +0.247      0.687      0.700      1.019
ls_5       -0.143     +0.236      0.673      0.682      1.013
ls_6       -0.065     +0.151      1.002      0.739      0.738
ls_7      +28.330     +2.530     10.828      2.585      0.239
ls_8       -0.567     -0.059      2.793      3.470      1.243
ls_9       -0.028     +0.099      2.738      4.134      1.510
ls_10      -0.011     -0.002      0.113      0.085      0.751
ls_11      -0.003     -0.002      0.109      0.081      0.747
ls_12      -0.016     -0.000      0.135      0.095      0.704
ls_13      +0.636     +0.438      0.386      0.544      1.411
ls_14      +0.711     +0.404    

In [9]:
#  [Cell 9 — Save logsigs_s4.parquet + small stats JSON]
df_out.to_parquet(PATH_OUT, index=False)
print(f"Saved: {PATH_OUT.name}  ({PATH_OUT.stat().st_size / 1e6:.2f} MB)")

stats = {
    'source_paths_file':   str(PATH_NSDE_PATHS.name),
    'cond_source_file':    str(PATH_LOGSIG_S3.name),
    'output_file':         str(PATH_OUT.name),
    'n_rows':              int(len(df_out)),
    'n_logsig_dims':       int(LOGSIG_DIM),
    'path_dim':            int(PATH_DIM),
    'sig_depth':           int(SIG_DEPTH),
    'segment_length':      int(SEGMENT_LENGTH),
    'use_lead_lag':        bool(USE_LEAD_LAG),
    'split_counts':        df_out['split'].value_counts().to_dict(),
    'logsig_per_feature_std': df_out[ls_cols].std().round(6).to_dict(),
}
with open(PATH_OUT_STATS, 'w') as f:
    json.dump(stats, f, indent=2, default=str)
print(f"Saved: {PATH_OUT_STATS.name}")

print("\nReady for Notebook 05 — point at:")
print(f"  REAL_FILE  = {PATH_LOGSIG_S3}")
print(f"  SYNTH_FILE = {PATH_OUT}")
print(f"  MODEL_NAME = 'neural_sde_s4'")

Saved: logsigs_s4.parquet  (6.51 MB)
Saved: logsigs_s4_stats.json

Ready for Notebook 05 — point at:
  REAL_FILE  = /content/drive/MyDrive/energy_synthetic_data/data/logsigs_s3.parquet
  SYNTH_FILE = /content/drive/MyDrive/energy_synthetic_data/data/logsigs_s4.parquet
  MODEL_NAME = 'neural_sde_s4'
